# 05 — Optuna Hyperparameter Tuning

Tuning the winning model from step 5 (XGBoost + `scale_pos_weight`, one-hot
encoded features). Bayesian search (TPE) with a median pruner, over the val AUC.

**Parallelism (10 cores available on this machine):** 5 concurrent Optuna trials
(`n_jobs=5`), each XGBoost fit using 2 threads (`n_jobs=2`) — 5×2=10, using the full
machine without oversubscribing any single core.

In [1]:
import sys, json, os, time
sys.path.append('..')

import pandas as pd
import optuna
from optuna.pruners import MedianPruner
from xgboost import XGBClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

from src.preprocessing import TARGET, CATEGORICAL_COLS, NUMERIC_COLS
from src.metrics import summarize

optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option('display.width', 120)

N_TRIAL_WORKERS = 5
N_JOBS_PER_MODEL = 2
N_TRIALS = 80

In [2]:
train = pd.read_csv('../data/processed/train.csv')
val = pd.read_csv('../data/processed/val.csv')
test = pd.read_csv('../data/processed/test.csv')

X_train, y_train = train.drop(columns=[TARGET]), train[TARGET]
X_val, y_val = val.drop(columns=[TARGET]), val[TARGET]
X_test, y_test = test.drop(columns=[TARGET]), test[TARGET]

ratio = (y_train == 0).sum() / (y_train == 1).sum()

encoder = ColumnTransformer([
    ('num', 'passthrough', NUMERIC_COLS),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False), CATEGORICAL_COLS),
], verbose_feature_names_out=False)
encoder.set_output(transform='pandas')

X_train_enc = encoder.fit_transform(X_train)
X_val_enc = encoder.transform(X_val)
X_test_enc = encoder.transform(X_test)

In [3]:
def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 200, 1000),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
    }
    model = XGBClassifier(
        **params, scale_pos_weight=ratio, eval_metric='auc',
        random_state=42, n_jobs=N_JOBS_PER_MODEL,
    )
    model.fit(X_train_enc, y_train)
    proba = model.predict_proba(X_val_enc)[:, 1]
    return summarize(y_val, proba)['auc']

In [4]:
study = optuna.create_study(direction='maximize', pruner=MedianPruner(), sampler=optuna.samplers.TPESampler(seed=42))

t0 = time.time()
study.optimize(objective, n_trials=N_TRIALS, n_jobs=N_TRIAL_WORKERS)
elapsed = time.time() - t0

print(f'{N_TRIALS} trials completed in {elapsed:.1f}s ({elapsed/N_TRIALS:.2f}s/trial average)')
print('best val AUC:', study.best_value)
print('best params:', study.best_params)

80 trials completed in 13.5s (0.17s/trial average)
best val AUC: 0.9493569492228671
best params: {'n_estimators': 535, 'learning_rate': 0.06519201262322011, 'max_depth': 6, 'subsample': 0.9333690229093594, 'colsample_bytree': 0.9488572466193919, 'min_child_weight': 3, 'reg_lambda': 6.688559472982023}


## Retrain with best params, evaluate on test — the honest, unseen-until-now split

In [5]:
best_model = XGBClassifier(
    **study.best_params, scale_pos_weight=ratio, eval_metric='auc',
    random_state=42, n_jobs=-1,
)
best_model.fit(X_train_enc, y_train)

proba_tuned_val = best_model.predict_proba(X_val_enc)[:, 1]
proba_tuned_test = best_model.predict_proba(X_test_enc)[:, 1]

default_results = json.load(open('../reports/xgboost_results.json'))['xgb_scale_pos_weight']['test']
comparison = pd.DataFrame({
    'xgb_optuna_tuned': summarize(y_test, proba_tuned_test),
    'xgb_default_params (step 5)': default_results,
}).T
print(comparison.round(4))

                                auc      ks    gini  pr_auc   brier
xgb_optuna_tuned             0.9484  0.7567  0.8968  0.9066  0.0616
xgb_default_params (step 5)  0.9459  0.7556  0.8919  0.9025  0.0681


In [6]:
os.makedirs('../reports', exist_ok=True)
os.makedirs('../models', exist_ok=True)

results = {
    'n_trials': N_TRIALS,
    'n_trial_workers': N_TRIAL_WORKERS,
    'n_jobs_per_model': N_JOBS_PER_MODEL,
    'elapsed_seconds': elapsed,
    'best_val_auc': float(study.best_value),
    'best_params': study.best_params,
    'test_metrics': summarize(y_test, proba_tuned_test),
    'val_metrics': summarize(y_val, proba_tuned_val),
}
with open('../reports/optuna_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=float)

import joblib
joblib.dump(best_model, '../models/xgb_tuned.pkl')
joblib.dump(encoder, '../models/encoder.pkl')
joblib.dump(list(X_train_enc.columns), '../models/feature_names.pkl')
print('saved reports/optuna_results.json, models/xgb_tuned.pkl, models/encoder.pkl, models/feature_names.pkl')

saved reports/optuna_results.json, models/xgb_tuned.pkl, models/encoder.pkl, models/feature_names.pkl


## Summary

- `N_TRIALS` Optuna trials run with TPE sampling + median pruning, 5 concurrent
  workers × 2 threads/model = full use of the machine's 10 cores.
- Tuned model compared honestly against the step-5 default-hyperparameter model on
  the untouched test split — actual numbers recorded above, not assumed.
- Best model, encoder, and feature name list saved to `models/` for the calibration
  (step 8) and SHAP (step 9) notebooks, and eventually the FastAPI service.